# Regressao Linear - Milho CEPEA ESALQ/B3 (Colab)

Este notebook replica o script `regressao_linear.py` para execucao no Google Colab.

## O que o modelo faz
- Usa duas fontes: medias anuais (`Media_Anual`) e cotacoes diarias (`Diario`).
- Concatena as series para capturar tendencia de longo + curto prazo.
- Treina dois modelos de regressao linear (BRL e USD).
- Preve preco para +3 e +6 dias uteis.
- Exibe metricas (R2, MAE), gera graficos e salva resultados no Excel.

## Observacao tecnica
Como a regressao e linear no tempo, ela funciona melhor para tendencia geral e pode nao capturar mudancas abruptas de mercado.

In [ ]:
# Se necessario, instale dependencias
!pip -q install openpyxl scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

DIAS_UTEIS_PREVISAO = [3, 6]

CORES = {
    "real_anual": "#9E9E9E",
    "real_diario": "#1565C0",
    "tendencia": "#E53935",
    "prev_3": "#F57C00",
    "prev_6": "#6A1B9A",
}

## 1) Carregar arquivo Excel
Se `cotacao_milho.xlsx` nao estiver em `/content`, o notebook pede upload.

In [ ]:
XLSX = '/content/cotacao_milho.xlsx'

if not os.path.exists(XLSX):
    from google.colab import files
    print('Arquivo nao encontrado em /content. Faca upload do cotacao_milho.xlsx...')
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise FileNotFoundError('Nenhum arquivo foi enviado.')
    nome_arquivo = next(iter(uploaded.keys()))
    XLSX = f'/content/{nome_arquivo}'

print(f'Usando arquivo: {XLSX}')

df_diario = pd.read_excel(XLSX, sheet_name='Diario', engine='openpyxl')
df_anual = pd.read_excel(XLSX, sheet_name='Media_Anual', engine='openpyxl')

df_diario['data'] = pd.to_datetime(df_diario['data'], errors='coerce')
df_anual['data'] = pd.to_datetime(df_anual['ano'].astype(str) + '-07-01', errors='coerce')

display(df_diario.head())
display(df_anual.head())

In [ ]:
# 2) Construir series combinadas
serie_rs = pd.concat([
    df_anual[['data', 'valor_rs_medio']].rename(columns={'valor_rs_medio': 'valor'}),
    df_diario[['data', 'valor_rs']].rename(columns={'valor_rs': 'valor'}),
], ignore_index=True).dropna().sort_values('data').reset_index(drop=True)

serie_usd = pd.concat([
    df_anual[['data', 'valor_usd_medio']].rename(columns={'valor_usd_medio': 'valor'}),
    df_diario[['data', 'valor_usd']].rename(columns={'valor_usd': 'valor'}),
], ignore_index=True).dropna().sort_values('data').reset_index(drop=True)

def treinar_modelo(serie: pd.DataFrame):
    data_ref = serie['data'].min()
    serie = serie.copy()
    serie['dias'] = (serie['data'] - data_ref).dt.days
    X = serie['dias'].values.reshape(-1, 1)
    y = serie['valor'].values
    modelo = LinearRegression()
    modelo.fit(X, y)
    return modelo, data_ref, serie

modelo_rs, data_ref_rs, serie_rs = treinar_modelo(serie_rs)
modelo_usd, data_ref_usd, serie_usd = treinar_modelo(serie_usd)

print('Modelos treinados com sucesso.')

In [ ]:
# 3) Previsoes e metricas
def proximos_dias_uteis(data_base: pd.Timestamp, n: int):
    datas = []
    d = data_base
    while len(datas) < n:
        d += pd.Timedelta(days=1)
        if d.weekday() < 5:
            datas.append(d)
    return datas

def metricas(modelo, serie):
    y_real = serie['valor'].values
    y_pred = modelo.predict(serie['dias'].values.reshape(-1, 1))
    return {
        'R2': round(r2_score(y_real, y_pred), 4),
        'MAE': round(mean_absolute_error(y_real, y_pred), 4),
        'Coef angular': round(float(modelo.coef_[0]), 6),
        'Intercepto': round(float(modelo.intercept_), 4),
    }

ultima_data = df_diario['data'].max()
previsoes = {}

for horizonte in DIAS_UTEIS_PREVISAO:
    datas_futuras = proximos_dias_uteis(ultima_data, horizonte)
    data_alvo = datas_futuras[-1]

    dias_rs = (data_alvo - data_ref_rs).days
    dias_usd = (data_alvo - data_ref_usd).days

    prev_rs = modelo_rs.predict(np.array([[dias_rs]]))[0]
    prev_usd = modelo_usd.predict(np.array([[dias_usd]]))[0]

    previsoes[horizonte] = {
        'data': data_alvo,
        'prev_rs': prev_rs,
        'prev_usd': prev_usd,
    }

m_rs = metricas(modelo_rs, serie_rs)
m_usd = metricas(modelo_usd, serie_usd)

print('=' * 55)
print('REGRESSAO LINEAR - MILHO CEPEA ESALQ/B3')
print('=' * 55)

print('\nMetricas BRL:')
for k, v in m_rs.items():
    print(f'  {k:15s}: {v}')

print('\nMetricas USD:')
for k, v in m_usd.items():
    print(f'  {k:15s}: {v}')

print('\nPrevisoes (saca 60kg):')
for h, p in previsoes.items():
    print(f'  +{h} dias uteis | {p["data"].strftime("%d/%m/%Y")} | R$ {p["prev_rs"]:.2f} | US$ {p["prev_usd"]:.2f}')

In [ ]:
# 4) Graficos
def plotar(serie, modelo, data_ref, previsoes, moeda, simbolo, cor_diario):
    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor('#FAFAFA')
    ax.set_facecolor('#FAFAFA')

    corte = df_diario['data'].min()
    anuais = serie[serie['data'] < corte]
    diarios = serie[serie['data'] >= corte]

    ax.scatter(anuais['data'], anuais['valor'], color=CORES['real_anual'], s=80, zorder=3, label='Medias anuais')
    ax.scatter(diarios['data'], diarios['valor'], color=cor_diario, s=60, zorder=4, label='Cotacoes diarias')

    x_line = np.linspace(serie['dias'].min(), serie['dias'].max(), 300)
    y_line = modelo.predict(x_line.reshape(-1, 1))
    datas_line = [data_ref + pd.Timedelta(days=int(d)) for d in x_line]
    ax.plot(datas_line, y_line, color=CORES['tendencia'], linewidth=2, label='Linha de tendencia')

    mapa_cores_prev = {3: CORES['prev_3'], 6: CORES['prev_6']}
    for h, p in previsoes.items():
        chave = 'prev_rs' if moeda == 'BRL' else 'prev_usd'
        val = p[chave]
        ax.scatter(p['data'], val, color=mapa_cores_prev[h], s=140, zorder=5, marker='D')
        ax.annotate(
            f'+{h}d\n{simbolo}{val:.2f}',
            xy=(p['data'], val),
            xytext=(10, 12),
            textcoords='offset points',
            fontsize=9,
            color=mapa_cores_prev[h],
            arrowprops=dict(arrowstyle='->', color=mapa_cores_prev[h]),
        )

    ax.axvline(df_diario['data'].max(), color='#757575', linestyle='--', linewidth=1, alpha=0.7, label='Hoje')

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=30, ha='right')

    X_plot = serie['dias'].values.reshape(-1, 1)
    titulo = (
        f'Milho CEPEA ESALQ/B3 - Previsao em {moeda}\n'
        f'R2 = {r2_score(serie["valor"], modelo.predict(X_plot)):.4f} | ' 
        f'MAE = {mean_absolute_error(serie["valor"], modelo.predict(X_plot)):.2f}'
    )
    ax.set_title(titulo, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Data')
    ax.set_ylabel(f'Preco ({simbolo} / saca 60 kg)')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    return fig

fig_rs = plotar(serie_rs, modelo_rs, data_ref_rs, previsoes, 'BRL', 'R$', CORES['real_diario'])
fig_usd = plotar(serie_usd, modelo_usd, data_ref_usd, previsoes, 'USD', 'US$', '#00695C')

plt.show()

fig_rs.savefig('/content/grafico_brl.png', dpi=150, bbox_inches='tight')
fig_usd.savefig('/content/grafico_usd.png', dpi=150, bbox_inches='tight')
print('Graficos salvos em /content/grafico_brl.png e /content/grafico_usd.png')

In [ ]:
# 5) Salvar metricas e previsoes no Excel e permitir download
df_metricas = pd.DataFrame([
    {'Modelo': 'BRL (R$)', **m_rs},
    {'Modelo': 'USD (US$)', **m_usd},
])

df_previsoes = pd.DataFrame([
    {
        'Horizonte (dias uteis)': h,
        'Data alvo': p['data'].strftime('%d/%m/%Y'),
        'Previsao R$ (BRL)': round(p['prev_rs'], 2),
        'Previsao US$ (USD)': round(p['prev_usd'], 2),
    }
    for h, p in previsoes.items()
])

saida_excel = '/content/cotacao_milho_atualizado.xlsx'

with pd.ExcelWriter(saida_excel, engine='openpyxl') as writer:
    # Copia as abas originais para manter o historico do arquivo
    df_diario.to_excel(writer, sheet_name='Diario', index=False)
    df_anual.drop(columns=['data'], errors='ignore').to_excel(writer, sheet_name='Media_Anual', index=False)
    df_metricas.to_excel(writer, sheet_name='Metricas', index=False)
    df_previsoes.to_excel(writer, sheet_name='Previsoes', index=False)

print(f'Arquivo salvo: {saida_excel}')

from google.colab import files
files.download(saida_excel)
files.download('/content/grafico_brl.png')
files.download('/content/grafico_usd.png')